In [1]:
from pathlib import Path

import pandas as pd


# Jupyter may start either in the repository root or in notebooks/.
CURRENT_DIR = Path.cwd().resolve()

if (CURRENT_DIR / "data").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. "
        "Start Jupyter from the repository root or notebooks/ directory."
    )

USGS_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / "usgs_gauge_metadata.csv"
)

MASTER_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "master_gauge_metadata.csv"
)

TIMESERIES_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "usgs"
    / "timeseries"
)

LONG_DATA_PATH = (
    TIMESERIES_DIR
    / "usgs_gauge_height_streamflow_long.csv"
)

WIDE_DATA_PATH = (
    TIMESERIES_DIR
    / "usgs_gauge_height_streamflow_wide.csv"
)

SUMMARY_DATA_PATH = (
    TIMESERIES_DIR
    / "usgs_download_summary.csv"
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Time-series directory: {TIMESERIES_DIR}")

Project root: /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone
Time-series directory: /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone/data/raw/usgs/timeseries


In [2]:
expected_files = {
    "USGS metadata": USGS_METADATA_PATH,
    "Master metadata": MASTER_METADATA_PATH,
    "Long time series": LONG_DATA_PATH,
    "Wide time series": WIDE_DATA_PATH,
    "Download summary": SUMMARY_DATA_PATH,
}

for name, path in expected_files.items():
    status = "found" if path.exists() else "missing"
    print(f"{name:20s} [{status}] {path}")

USGS metadata        [found] /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone/data/interim/usgs_gauge_metadata.csv
Master metadata      [found] /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone/data/processed/master_gauge_metadata.csv
Long time series     [found] /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone/data/raw/usgs/timeseries/usgs_gauge_height_streamflow_long.csv
Wide time series     [found] /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone/data/raw/usgs/timeseries/usgs_gauge_height_streamflow_wide.csv
Download summary     [found] /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone/data/raw/usgs/timeseries/usgs_download_summary.csv


In [3]:
usgs_metadata = pd.read_csv(
    USGS_METADATA_PATH,
    dtype={
        "site_id": str,
        "huc": str,
    },
)

usgs_metadata["site_id"] = (
    usgs_metadata["site_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(8)
)

print(f"Shape: {usgs_metadata.shape}")
print("\nColumns:")
print(usgs_metadata.columns.tolist())

display(usgs_metadata.head())

print("\nData types and non-null counts:")
usgs_metadata.info()

Shape: (10, 14)

Columns:
['site_id', 'site_name', 'latitude', 'longitude', 'site_type', 'state', 'county', 'huc', 'basin_code', 'altitude_ft', 'vertical_datum', 'drainage_area_sqmi', 'contributing_area_sqmi', 'time_zone']


,site_id,site_name,latitude,longitude,site_type,state,county,huc,basin_code,altitude_ft,vertical_datum,drainage_area_sqmi,contributing_area_sqmi,time_zone
0,08193000,"Nueces Rv nr Asherton, TX",28.500263,-99.681993,Stream,Texas,Dimmit County,121101030306,NaN,470.53,North American Vertical Datum of 1988,4082.0,4082.0,CST
1,08194000,"Nueces Rv at Cotulla, TX",28.426379,-99.240032,Stream,Texas,La Salle County,121101050201,NaN,368.19,North American Vertical Datum of 1988,5171.0,5171.0,CST
2,08194500,"Nueces Rv nr Tilden, TX",28.308889,-98.557238,Stream,Texas,McMullen County,121101051105,NaN,183.10,North American Vertical Datum of 1988,8093.0,8093.0,CST
3,08205500,"Frio Rv nr Derby, TX",28.736644,-99.144756,Stream,Texas,Frio County,121101061205,NaN,449.05,North American Vertical Datum of 1988,3429.0,3429.0,CST
4,08206600,"Frio Rv at Tilden, TX",28.467493,-98.547517,Stream,Texas,McMullen County,121101080501,NaN,212.66,North American Vertical Datum of 1988,4493.0,4493.0,CST



Data types and non-null counts:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   site_id                 10 non-null     string 
 1   site_name               10 non-null     str    
 2   latitude                10 non-null     float64
 3   longitude               10 non-null     float64
 4   site_type               10 non-null     str    
 5   state                   10 non-null     str    
 6   county                  10 non-null     str    
 7   huc                     10 non-null     str    
 8   basin_code              0 non-null      float64
 9   altitude_ft             10 non-null     float64
 10  vertical_datum          10 non-null     str    
 11  drainage_area_sqmi      10 non-null     float64
 12  contributing_area_sqmi  10 non-null     float64
 13  time_zone               10 non-null     str    
dtypes: float64(6), str(7), 

In [4]:
master_metadata = pd.read_csv(
    MASTER_METADATA_PATH,
    dtype={
        "site_id": str,
        "reach_id": str,
        "huc": str,
    },
)

master_metadata["site_id"] = (
    master_metadata["site_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(8)
)

print(f"Shape: {master_metadata.shape}")

display(master_metadata)

Shape: (10, 26)


,site_id,reach_id,site_name,river,site_category,latitude,longitude,site_type,state,county,...,action_stage_ft,minor_flood_stage_ft,moderate_flood_stage_ft,major_flood_stage_ft,gauge_status,has_reach_id,has_flood_categories,flood_thresholds_valid,notes,basin_code
0,08208000,10828654,"Atascosa Rv at Whitsett, TX",Atascosa River,Stream,28.622209,-98.281399,Stream,Texas,Live Oak County,...,20.0,20.0,24.0,26.0,active,True,True,True,NOAA reports the same threshold for action and...,NaN
1,08206900,NaN,"Choke Canyon Res nr Three Rivers, TX",Choke Canyon Reservoir,Reservoir,28.483881,-98.245842,"Lake, Reservoir, Impoundment",Texas,Live Oak County,...,NaN,NaN,NaN,NaN,active,False,False,NaN,No flood-stage category data shown on NOAA NWPS.,NaN
2,08205500,10661028,"Frio Rv nr Derby, TX",Frio River,Stream,28.736644,-99.144756,Stream,Texas,Frio County,...,6.0,6.0,7.0,17.0,active,True,True,True,NOAA reports the same threshold for action and...,NaN
3,08206600,10664196,"Frio Rv at Tilden, TX",Frio River,Stream,28.467493,-98.547517,Stream,Texas,McMullen County,...,12.0,22.0,26.0,27.0,active,True,True,True,NaN,NaN
4,08193000,10893541,"Nueces Rv nr Asherton, TX",Nueces River,Stream,28.500263,-99.681993,Stream,Texas,Dimmit County,...,18.0,20.0,24.0,27.0,active,True,True,True,NaN,NaN
5,08194000,10630401,"Nueces Rv at Cotulla, TX",Nueces River,Stream,28.426379,-99.240032,Stream,Texas,La Salle County,...,9.0,15.0,15.0,17.0,active,True,True,True,NOAA reports the same threshold for minor and ...,NaN
6,08194500,10631613,"Nueces Rv nr Tilden, TX",Nueces River,Stream,28.308889,-98.557238,Stream,Texas,McMullen County,...,11.0,14.0,16.0,19.0,active,True,True,True,NaN,NaN
7,08210000,3168766,"Nueces Rv nr Three Rivers, TX",Nueces River,Stream,28.427495,-98.178063,Stream,Texas,Live Oak County,...,20.0,25.0,27.0,35.0,active,True,True,True,NaN,NaN
8,08210100,NaN,"Nueces Rv at George West, TX",Nueces River,Stream,28.332778,-98.085556,Stream,Texas,Live Oak County,...,NaN,NaN,NaN,NaN,discontinued,False,False,NaN,Gauge discontinued after 2010.,NaN
9,08206700,10671061,"San Miguel Ck nr Tilden, TX",San Miguel Creek,Stream,28.587488,-98.545851,Stream,Texas,McMullen County,...,12.0,21.0,23.0,26.0,active,True,True,True,NaN,NaN


In [5]:
# %%
wide_df = pd.read_csv(
    WIDE_DATA_PATH,
    dtype={
        "site_id": str,
        "reach_id": str,
    },
    parse_dates=["datetime"],
)

wide_df["site_id"] = (
    wide_df["site_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(8)
)

print(f"Shape: {wide_df.shape}")

print("\nColumns:")
print(wide_df.columns.tolist())

display(wide_df.head())

Shape: (4598644, 16)

Columns:
['site_id', 'site_name', 'river', 'site_category', 'reach_id', 'datetime', 'streamflow_cfs', 'gage_height_ft', 'latitude', 'longitude', 'drainage_area_sqmi', 'action_stage_ft', 'minor_flood_stage_ft', 'moderate_flood_stage_ft', 'major_flood_stage_ft', 'gauge_status']


,site_id,site_name,river,site_category,reach_id,datetime,streamflow_cfs,gage_height_ft,latitude,longitude,drainage_area_sqmi,action_stage_ft,minor_flood_stage_ft,moderate_flood_stage_ft,major_flood_stage_ft,gauge_status
0,08193000,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,2010-01-01 06:00:00+00:00,0.0,0.32,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
1,08193000,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,2010-01-01 06:15:00+00:00,0.0,0.31,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
2,08193000,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,2010-01-01 06:30:00+00:00,0.0,0.32,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
3,08193000,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,2010-01-01 06:45:00+00:00,0.0,0.31,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
4,08193000,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,2010-01-01 07:00:00+00:00,0.0,0.31,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active


In [6]:
date_coverage = (
    wide_df.groupby("site_id", dropna=False)
    .agg(
        first_datetime=("datetime", "min"),
        last_datetime=("datetime", "max"),
        row_count=("datetime", "size"),
    )
    .reset_index()
)

date_coverage["coverage_days"] = (
    date_coverage["last_datetime"]
    - date_coverage["first_datetime"]
).dt.days + 1

display(date_coverage)

,site_id,first_datetime,last_datetime,row_count,coverage_days
0,08193000,2010-01-01 06:00:00+00:00,2026-07-27 19:30:00+00:00,579225,6052
1,08194000,2010-01-01 06:00:00+00:00,2026-07-27 20:00:00+00:00,568716,6052
2,08194500,2010-01-01 06:00:00+00:00,2026-07-27 19:30:00+00:00,574853,6052
3,08205500,2010-01-06 06:00:00+00:00,2026-07-27 19:30:00+00:00,575077,6047
4,08206600,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,570896,6052
5,08206700,2010-01-01 06:00:00+00:00,2026-07-27 19:15:00+00:00,578088,6052
6,08208000,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,576797,6052
7,08210000,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,574992,6052


In [7]:
measurement_columns = [
    column
    for column in [
        "streamflow_cfs",
        "gage_height_ft",
    ]
    if column in wide_df.columns
]

missing_summary = pd.DataFrame(
    {
        "missing_count": wide_df[measurement_columns].isna().sum(),
        "available_count": wide_df[measurement_columns].notna().sum(),
        "missing_percent": (
            wide_df[measurement_columns].isna().mean() * 100
        ).round(2),
    }
)

display(missing_summary)

,missing_count,available_count,missing_percent
streamflow_cfs,3322,4595322,0.07
gage_height_ft,6070,4592574,0.13


In [8]:
missing_by_gauge = (
    wide_df.groupby("site_id")[measurement_columns]
    .agg(
        lambda series: series.isna().mean() * 100
    )
    .round(2)
    .reset_index()
)

missing_by_gauge = missing_by_gauge.rename(
    columns={
        "streamflow_cfs": "streamflow_missing_percent",
        "gage_height_ft": "gage_height_missing_percent",
    }
)

display(missing_by_gauge)

,site_id,streamflow_missing_percent,gage_height_missing_percent
0,08193000,0.00,0.10
1,08194000,0.56,0.40
2,08194500,0.00,0.04
3,08205500,0.00,0.00
4,08206600,0.00,0.23
5,08206700,0.00,0.03
6,08208000,0.02,0.17
7,08210000,0.01,0.10


In [9]:
from pathlib import Path

import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

ROOT = Path("..").resolve()        # if running from notebooks/
TS_DIR = ROOT / "data" / "raw" / "usgs" / "timeseries"

META = ROOT / "data" / "processed" / "master_gauge_metadata.csv"

LONG = TS_DIR / "usgs_gauge_height_streamflow_long.csv"

# ------------------------------------------------------------------

meta = pd.read_csv(META)

df = pd.read_csv(
    LONG,
    parse_dates=["datetime"]
)

print(df.shape)
display(df.head())

(9187896, 20)


,site_id,site_name_api,datetime,parameter_code,parameter_name,value,unit,qualifiers,site_name,river,site_category,reach_id,latitude,longitude,drainage_area_sqmi,action_stage_ft,minor_flood_stage_ft,moderate_flood_stage_ft,major_flood_stage_ft,gauge_status
0,8193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:00:00+00:00,60,streamflow_cfs,0.00,ft3/s,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
1,8193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:00:00+00:00,65,gage_height_ft,0.32,ft,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
2,8193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:15:00+00:00,60,streamflow_cfs,0.00,ft3/s,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
3,8193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:15:00+00:00,65,gage_height_ft,0.31,ft,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
4,8193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:30:00+00:00,60,streamflow_cfs,0.00,ft3/s,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active


In [10]:
print(df.columns.tolist())

['site_id', 'site_name_api', 'datetime', 'parameter_code', 'parameter_name', 'value', 'unit', 'qualifiers', 'site_name', 'river', 'site_category', 'reach_id', 'latitude', 'longitude', 'drainage_area_sqmi', 'action_stage_ft', 'minor_flood_stage_ft', 'moderate_flood_stage_ft', 'major_flood_stage_ft', 'gauge_status']


In [12]:
print("Number of gauges:", df.site_id.nunique())

display(
    df.groupby("site_id")
      .size()
      .rename("observations")
      .to_frame()
)

Number of gauges: 8


,observations
site_id,
8193000,1157894
8194000,1132009
8194500,1149454
8205500,1150152
8206600,1140466
8206700,1156030
8208000,1152545
8210000,1149346


In [13]:
summary = (
    df.groupby("site_id")
      .agg(
          start=("datetime", "min"),
          end=("datetime", "max"),
          observations=("datetime", "count"),
      )
)

summary["years"] = (
    (summary["end"] - summary["start"])
    .dt.days
    / 365.25
)

summary

,start,end,observations,years
site_id,,,,
8193000,2010-01-01 06:00:00+00:00,2026-07-27 19:30:00+00:00,1157894,16.566735
8194000,2010-01-01 06:00:00+00:00,2026-07-27 20:00:00+00:00,1132009,16.566735
8194500,2010-01-01 06:00:00+00:00,2026-07-27 19:30:00+00:00,1149454,16.566735
8205500,2010-01-06 06:00:00+00:00,2026-07-27 19:30:00+00:00,1150152,16.553046
8206600,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,1140466,16.566735
8206700,2010-01-01 06:00:00+00:00,2026-07-27 19:15:00+00:00,1156030,16.566735
8208000,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,1152545,16.566735
8210000,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,1149346,16.566735


In [14]:
summary = (
    summary
    .reset_index()
    .merge(
        meta[
            [
                "site_id",
                "river",
                "site_name",
                "drainage_area_sqmi",
                "altitude_ft",
                "action_stage_ft",
                "minor_flood_stage_ft",
                "moderate_flood_stage_ft",
                "major_flood_stage_ft",
            ]
        ],
        on="site_id",
        how="left",
    )
)

summary

,site_id,start,end,observations,years,river,site_name,drainage_area_sqmi,altitude_ft,action_stage_ft,minor_flood_stage_ft,moderate_flood_stage_ft,major_flood_stage_ft
0,8193000,2010-01-01 06:00:00+00:00,2026-07-27 19:30:00+00:00,1157894,16.566735,Nueces River,"Nueces Rv nr Asherton, TX",4082.0,470.53,18.0,20.0,24.0,27.0
1,8194000,2010-01-01 06:00:00+00:00,2026-07-27 20:00:00+00:00,1132009,16.566735,Nueces River,"Nueces Rv at Cotulla, TX",5171.0,368.19,9.0,15.0,15.0,17.0
2,8194500,2010-01-01 06:00:00+00:00,2026-07-27 19:30:00+00:00,1149454,16.566735,Nueces River,"Nueces Rv nr Tilden, TX",8093.0,183.10,11.0,14.0,16.0,19.0
3,8205500,2010-01-06 06:00:00+00:00,2026-07-27 19:30:00+00:00,1150152,16.553046,Frio River,"Frio Rv nr Derby, TX",3429.0,449.05,6.0,6.0,7.0,17.0
4,8206600,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,1140466,16.566735,Frio River,"Frio Rv at Tilden, TX",4493.0,212.66,12.0,22.0,26.0,27.0
5,8206700,2010-01-01 06:00:00+00:00,2026-07-27 19:15:00+00:00,1156030,16.566735,San Miguel Creek,"San Miguel Ck nr Tilden, TX",783.0,242.56,12.0,21.0,23.0,26.0
6,8208000,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,1152545,16.566735,Atascosa River,"Atascosa Rv at Whitsett, TX",1171.0,158.71,20.0,20.0,24.0,26.0
7,8210000,2010-01-01 06:00:00+00:00,2026-07-27 20:15:00+00:00,1149346,16.566735,Nueces River,"Nueces Rv nr Three Rivers, TX",15427.0,166.31,20.0,25.0,27.0,35.0


In [16]:
from pathlib import Path
import pandas as pd

# Notebook is located in WaterSoft26_Capstone/notebooks/
ROOT = Path("..").resolve()

long_path = (
    ROOT
    / "data"
    / "raw"
    / "usgs"
    / "timeseries"
    / "usgs_gauge_height_streamflow_long.csv"
)

print("Project root:", ROOT)
print("Input file:", long_path)
print("File exists:", long_path.exists())

df = pd.read_csv(
    long_path,
    parse_dates=["datetime"],
    dtype={"site_id": "string"},
)

df["site_id"] = df["site_id"].str.zfill(8)
df["value"] = pd.to_numeric(df["value"], errors="coerce")

display(df.head())

Project root: /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone
Input file: /Users/omidzandi/Desktop/PhD/WaterSoftHack/2026/WaterSoft26_Capstone/data/raw/usgs/timeseries/usgs_gauge_height_streamflow_long.csv
File exists: True


,site_id,site_name_api,datetime,parameter_code,parameter_name,value,unit,qualifiers,site_name,river,site_category,reach_id,latitude,longitude,drainage_area_sqmi,action_stage_ft,minor_flood_stage_ft,moderate_flood_stage_ft,major_flood_stage_ft,gauge_status
0,08193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:00:00+00:00,60,streamflow_cfs,0.00,ft3/s,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
1,08193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:00:00+00:00,65,gage_height_ft,0.32,ft,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
2,08193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:15:00+00:00,60,streamflow_cfs,0.00,ft3/s,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
3,08193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:15:00+00:00,65,gage_height_ft,0.31,ft,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
4,08193000,"Nueces Rv nr Asherton, TX",2010-01-01 06:30:00+00:00,60,streamflow_cfs,0.00,ft3/s,A,"Nueces Rv nr Asherton, TX",Nueces River,Stream,10893541,28.500263,-99.681993,4082.0,18.0,20.0,24.0,27.0,active
